# MJO RMM/EOF diagnostics

Compute ensemble RMM projections, skill scores, phase-space figures, and amplitude distributions. This is the maintained replacement for `7_mjo_analysis/mjo_analysis.ipynb`.


In [ ]:
# =============================================================================
# TOP-LEVEL PATH PARAMETERS
# Edit these three paths to relocate inputs, the analysis checkout, or outputs.
# =============================================================================
from pathlib import Path
import sys

INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_out")
WORKFLOW_NAME = "analysis_mjo"

DIAGNOSTIC_DIR = WORK_DIR / "diagnostic"
if not DIAGNOSTIC_DIR.is_dir():
    raise FileNotFoundError(f"Diagnostic work directory not found: {DIAGNOSTIC_DIR}")
if str(DIAGNOSTIC_DIR) not in sys.path:
    sys.path.insert(0, str(DIAGNOSTIC_DIR))

from configs.output_paths import workflow_output_dirs
WORKFLOW_DATA_DIR, WORKFLOW_FIGURE_DIR = workflow_output_dirs(
    WORKFLOW_NAME,
    output_root=DIAGNOSTIC_OUTPUT_ROOT,
)


## Analysis configuration


In [ ]:
COMPSET = "F20TR"
RESOLUTION = "ne30pg2_r05_IcoswISC30E3r5"
MACHINE = "compy"
EXPERIMENTS = ["CAPTEN10_15day", "DARTEN20_15day", "DARTEN40_15day"]
FREQUENCY = "daily"
REGION_NAME = "Tropics"
ANALYSIS_YEARS = [2012]
OVERWRITE_EOFS = False


## Run diagnostics


In [ ]:
from util.mjo_model_data import ModelDataReader
from util.mjo_rmm_diagnostics import MJODiagnosticAnalyzer

experiment_suffix = f"{COMPSET}_{RESOLUTION}_{MACHINE}"
rmm_input_dir = INPUT_DATA_ROOT / "Observations/MJO_INDEX"
rmm_cache_dir = WORKFLOW_DATA_DIR / "rmm_index"

reader = ModelDataReader(
    base_path=str(INPUT_DATA_ROOT),
    exp_base=experiment_suffix,
    regnam=REGION_NAME,
    frequency=FREQUENCY,
    component="atm",
)

for experiment in EXPERIMENTS:
    print(f"\n[INFO] Running RMM diagnostics for {experiment}")
    data_dir = WORKFLOW_DATA_DIR / experiment
    figure_dir = WORKFLOW_FIGURE_DIR / experiment
    data_dir.mkdir(parents=True, exist_ok=True)
    figure_dir.mkdir(parents=True, exist_ok=True)

    olr = reader.read_variable("FLUT", experiment)
    u850 = reader.read_variable("U850", experiment)
    v850 = reader.read_variable("V850", experiment)
    u200 = reader.read_variable("U200", experiment)
    v200 = reader.read_variable("V200", experiment)

    analyzer = MJODiagnosticAnalyzer(
        mjoindx_dir=str(rmm_input_dir),
        rmm_cache_dir=str(rmm_cache_dir),
        olr_data=olr,
        u850_data=u850,
        v850_data=v850,
        u200_data=u200,
        v200_data=v200,
        time=olr.time.to_pandas(),
        frequency=FREQUENCY,
        output_dir=str(data_dir),
        figure_dir=str(figure_dir),
        region_bounds=(-15, 15),
        years=ANALYSIS_YEARS,
        reload_mjo=False,
        overwrite_eofs=OVERWRITE_EOFS,
    )
    results = analyzer.analyze_all_members()
    print(f"[INFO] Saved {len(results)} member diagnostics for {experiment}")
